In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
import json
from models.llm_ner import get_entities_from_llm
from models.llm_text_embeddings import get_embeddings
from models.metric import CDE, exhaustive_CDE, EF, CDEF

In [3]:
def parse_response(response:str):
    response = json.loads(response)
    
    entities = []

    for e in response['entities']:
        entity = e['entity']
        type = e['type']
        entities.append([entity, type])

    return entities

# Use Cases
Three types of entities:
- Symptom,
- Drug,
- Disease.

In [4]:
types = ['Symptom', 'Drug', 'Disease']

## 1. Discontinuous Entities

### UC1.1 
**The patient reported pain in the lower back and occasionally in the right leg.**
- pain in the lower back; Symptom
- pain in the right leg; Symptom

In [5]:
sentence = "The patient reported pain in the lower back and occasionally in the right leg."
gold_entities = [
    ['pain in the lower back', 'Symptom'],
    ['pain in the right leg', 'Symptom']
]

In [6]:
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
generated_entities

[['pain', 'Symptom'], ['lower back', 'Symptom'], ['right leg', 'Symptom']]

In [18]:
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])

In [10]:
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')

1, 2, 2, 768


In [17]:
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

CDE:		0.1215967271289205
Exh_CDE:	0.1215967271289205
EF:		0.19999999999999996
CDEF:		0.8640301313059162


### UC1.2
**The patient experienced shortness of breath on exertion and sometimes at rest.**
- shortness of breath on exertion; Symptom
- shortness of breath at rest; Symptom

In [20]:
sentence = "The patient experienced shortness of breath on exertion and sometimes at rest."
gold_entities = [
    ['shortness of breath on exertion', 'Symptom'],
    ['shortness of breath at rest', 'Symptom']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['shortness of breath', 'Symptom']]
1, 2, 2, 768
CDE:		0.11803836561393666
Exh_CDE:	0.11803836561393666
EF:		-0.33333333333333337
CDEF:		0.7804205226499786


### UC1.2
**Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better.**
- muscle pain; Symptom
- joint pain; Symptom

In [21]:
sentence = "Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better."
gold_entities = [
    ['muscle pain', 'Symptom'],
    ['joint pain', 'Symptom']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['muscle/joint pain', 'Symptom']]
1, 2, 2, 768
CDE:		0.10130145729433637
Exh_CDE:	0.10130145729433637
EF:		-0.33333333333333337
CDEF:		0.7832837527714838


### UC1.3
**Menstrual cramps present with or without vaginal bleeding.**
- menstrual cramps with vaginal bleeding; Symptom
- menstrual cramps without vaginal bleeding; Symptom

In [22]:
sentence = "Menstrual cramps present with or without vaginal bleeding."
gold_entities = [
    ['menstrual cramps with vaginal bleeding', 'Symptom'],
    ['menstrual cramps without vaginal bleeding', 'Symptom']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['menstrual cramps', 'Symptom'], ['vaginal bleeding', 'Symptom']]
1, 2, 2, 768
CDE:		0.14882194156765166
Exh_CDE:	0.14882194156765166
EF:		0.0
CDEF:		0.9613567746519021


### UC1.4
**After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day.**
- abdominal gas; Symptom
- abdominal cramps; Symptom
- abdominal pain; Symptom

In [24]:
sentence = "After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day."
gold_entities = [
    ['abdominal gas', 'Symptom'],
    ['abdominal cramps', 'Symptom'],
    ['abdominal pain', 'Symptom']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['abdominal gas', 'Symptom'], ['cramps', 'Symptom'], ['pain', 'Symptom']]
1, 3, 2, 768
CDE:		0.1263790903944034
Exh_CDE:	0.1263790903944034
EF:		0.0
CDEF:		0.9673744299342727


## 2. Long, Descriptive Entities

### UC2.1
**Without this drug I was severly restricted and could only walk less than 100 meters.**
- could only walk less than 100 meters; Symptom

In [25]:
sentence = "Without this drug I was severly restricted and could only walk less than 100 meters."
gold_entities = [
    ['could only walk less than 100 meters', 'Symptom']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['this drug', 'Drug']]
1, 1, 2, 768
CDE:		2.0
Exh_CDE:	2.0
EF:		0.0
CDEF:		0.0


### UC2.2
**Recently I've been feeling like my stomach is full and empty at the same time hard to explain.**
- feeling stomach is full and empty at the same time; Symptom

In [26]:
sentence = "Recently I've been feeling like my stomach is full and empty at the same time hard to explain."
gold_entities = [
    ['feeling stomach is full and empty at the same time', 'Symptom']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['stomach', 'Disease'], ['full', 'Symptom'], ['empty', 'Symptom']]
1, 1, 2, 768
CDE:		0.3367578766208573
Exh_CDE:	0.3367578766208573
EF:		0.5
CDEF:		0.6245178043627546


### UC2.3
**The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung.**
- lung infection of the lower part of right lung; Disease

In [27]:
sentence = "The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung."
gold_entities = [
    ['lung infection of the lower part of right lung', 'Disease']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['lung infection', 'Disease']]
1, 1, 2, 768
CDE:		0.18487491599217287
Exh_CDE:	0.18487491599217287
EF:		0.0
CDEF:		0.9515415846345043


### UC2.4
**The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself.**
- strong intravenous medicine; Drug
- swelling in the brain; Disease
- immune system attacking itself; Disease

In [28]:
sentence = "The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself."
gold_entities = [
    ['strong intravenous medicine', 'Drug'],
    ['swelling in the brain', 'Disease'],
    ['immune system attacking itself', 'Disease']
]
response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
print(generated_entities)
gold_embeddings = get_embeddings([gold_entities])
generated_embeddings = get_embeddings([generated_entities])
print(f'{len(gold_embeddings)}, {len(gold_embeddings[0])}, {len(gold_embeddings[0][0])}, {len(gold_embeddings[0][0][0])}')
for a, b in zip(gold_embeddings, generated_embeddings):
    print(f'CDE:\t\t{CDE(a, b)}')
    print(f'Exh_CDE:\t{exhaustive_CDE(a, b)}')
    print(f'EF:\t\t{EF(a, b)}')
    print(f'CDEF:\t\t{CDEF(a, b)}')

[['swelling', 'Symptom'], ['strong medicine', 'Drug'], ['the immune system attacking itself', 'Disease']]
1, 3, 2, 768
CDE:		0.7233145943836132
Exh_CDE:	0.6666666666666666
EF:		0.0
CDEF:		0.7792541837724735
